# Polarized Trees — How the PEG Formulations Change the Trees

Fourth notebook of the synthetic workflow: `datasetdemo.ipynb` (data) → `treesbenchmark.ipynb` (search, saves the top 20) → `resultsexploration.ipynb` (explores those results) → **this notebook**.

The comparison itself is `polartox.peg_comparison` (`PEGComparison`): this notebook chooses the settings, calls it and displays the tables.

We take settings from the top 20, keep **everything fixed except the PEG formulation** (`max`, `avg`, `min`, `mean`, `harmonic`), and look at what the formulation changes:

1. **Overview**: recovery and tree shape (leaves, depth, leaf size, residual polarization) for each formulation.
2. **By k**: the same numbers split by the true number of active dimensions of the text, where the formulations differ most.
3. **ARI**: do two formulations split the annotators of a text into the same groups?
4. **F, C, P** on the unseen corpus.
5. **The same text, different trees**: the texts on which the formulations disagree most.

> Run `datasetdemo.ipynb` and `treesbenchmark.ipynb` first, so that `benchmark_data/` and `benchmark_results/top_configurations.csv` exist.

## 1. Setup

In [6]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


# Locate the benchmarks folder (it holds benchmark_config.py).
CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Run this notebook from the notebooks directory "
        "or the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from benchmark_config import (
    DIMS,
    INFERENCE_CORPUS,
    RESULTS_DIR,
    SCALE,
    check_polartox,
    load_benchmark_corpora,
    load_corpus,
)

check_polartox()    # which polartox and which library versions produce these results

from polartox import PEGComparison, PolarizedTreesPipeline, pairwise_ari
from polartox.peg_comparison import TREE
from polartox.polarized_tree import PEG_VARIANTS

# Everything this notebook saves goes here, never over the benchmark notebook's results.
OUT_DIR = RESULTS_DIR / "peg_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# A, B, C have ground truth: the formulations are compared on them. The unseen corpus has none
# and is only looked at (trees, ARI, F, C, P).
corpora = load_benchmark_corpora()
unseen, _ = load_corpus(INFERENCE_CORPUS)

for name, (data, truth) in corpora.items():
    print(f"{name}: {len(data):,} annotations, {data['text_id'].nunique()} texts")
print(f"{INFERENCE_CORPUS}: {len(unseen):,} annotations, {unseen['text_id'].nunique()} texts (no ground truth used)")

polartox 0.8.0 imported from C:\Users\swkra\OneDrive\Υπολογιστής\ερευνα\HumanModelAlign\polarizedtrees\polartox\__init__.py
PEG formulations: ('max', 'avg', 'min', 'mean', 'harmonic')
numpy 2.5.3
pandas 3.0.6
ndfu 0.9.3
scikit-learn 1.9.1
A_default: 162,000 annotations, 100 texts
B_weak_signal: 162,000 annotations, 100 texts
C_deep: 162,000 annotations, 100 texts
inference_unseen: 162,000 annotations, 100 texts (no ground truth used)


## 2. Choose the settings

`top_configurations.csv` holds the 20 best configurations of the search; `resultsexploration.ipynb` shows them in full (every metric, per corpus). Put the row positions you want in `SELECTED`: each chosen row gives the non-PEG settings, and its own `variant` is **ignored** (every formulation is run with those settings). Choose several rows to compare the formulations at several settings.

In [ ]:
top = pd.read_csv(RESULTS_DIR / "top_configurations.csv")

NON_PEG = ["theta_filter", "min_size_frac", "max_depth", "h", "relative_h", "theta_stop"]
display(top[["rank", "variant", *NON_PEG, "jaccard", "exact_match"]].round(4))

SELECTED = [0]      # row positions in the table above

## 3. Run every formulation on every corpus

One pipeline per (chosen row, formulation, corpus); `beta` keeps its default of 1.

In [3]:
comparison = PEGComparison(corpora, dims=DIMS, scale=SCALE, variants=PEG_VARIANTS)
comparison.run(top.loc[SELECTED])       # the non-PEG settings of the chosen rows, labelled by row position

# Sanity check: with its own formulation, a chosen row must reproduce what the benchmark saved.
checked = comparison.check_against_benchmark(top)
print(f"OK: the chosen rows reproduce all {checked} numbers the benchmark saved for them.")

# The unseen corpus has no ground truth, so it is run on its own, with the same settings.
unseen_runs = {}        # (row, formulation) -> (pipeline, results)
for position, settings in comparison.settings_.items():
    for variant in PEG_VARIANTS:
        pipe = PolarizedTreesPipeline(dims=DIMS, scale=SCALE, variant=variant, **settings)
        unseen_runs[position, variant] = (pipe, pipe.run_full_evaluation(unseen, verbose=False))

OK: the chosen rows reproduce all 22 numbers the benchmark saved for them.


## 4. Overview

One row per formulation, everything else fixed.

- **Recovery** (A/B/C only, where the truth is known): mean and median Jaccard, precision, recall, exact match.
- **Trees**: *Retention* is the share of texts polarized enough to be analysed, *Leaves* and *Depth* are the means per tree (depth is the mean over leaves), *Annotators per leaf* is the mean number of annotators in a leaf (not the number of leaves, which is *Leaves*), taken within each tree and then averaged over the trees, *Residual nDFU* is the polarization left in the leaves, *Top-split PEG* is the mean PEG of the first split, and *Indeterminate* is the share of leaves with no majority pole.

The A/B/C numbers are averaged with equal weight per corpus. The unseen corpus has no ground truth, so only the trees are shown.

In [ ]:
SHOWN = {"jaccard": "Jaccard", "jaccard_median": "Jaccard median", "precision": "Precision",
         "recall": "Recall", "exact_match": "Exact match", **TREE}
overview = comparison.overview()

for position in SELECTED:
    print(f"=== top row {position}: mean over A/B/C ===")
    display(overview.loc[position, list(SHOWN)].rename(columns=SHOWN).rename_axis(None).round(3))

    print(f"=== top row {position}: unseen corpus (no ground truth) ===")
    rows = {}
    for variant in PEG_VARIANTS:
        pipe, results = unseen_runs[position, variant]
        rows[variant] = {
            **{key: results["diagnostics"][key] for key in TREE if key != "mean_leaf_size"},
            "mean_leaf_size": np.mean([np.mean([leaf["n"] for leaf in tree.get_leaves()]) for tree in pipe.trees_.values()]),
        }
    display(pd.DataFrame(rows).T[list(TREE)].rename(columns=TREE).round(3))

### By the true number of active dimensions (k)

The overview averages over texts with one, two, three or four true causes of polarization. Split by that number, the formulations behave very differently. Pooled over the texts of A, B and C. *Precision* low means extra dimensions were added (over-splitting), *recall* low means true dimensions were missed (under-splitting).

In [ ]:
recovery_by_k = comparison.recovery_by_k()

for position in SELECTED:
    table = recovery_by_k.loc[position]
    print(f"texts per k (top row {position}):", table.xs(PEG_VARIANTS[0], level="variant")["texts"].to_dict())
    for metric in ("jaccard", "exact_match", "precision", "recall"):
        print(f"=== top row {position}: mean {metric} by k ===")
        display(table[metric].unstack("variant")[list(PEG_VARIANTS)].rename_axis(None, axis=1).round(3))

## 5. Do the formulations build the same trees? (ARI)

The leaves of a tree split the annotators of a text into groups. The **adjusted Rand index** compares two such splits of the same annotators: **1** means the same groups, **0** means as different as chance. Below, the mean over the texts that all formulations analysed.

In [ ]:
unseen_ari = {}     # row -> (mean ARI matrix, ARI per text) on the unseen corpus
for position in SELECTED:
    print(f"=== top row {position}: mean ARI over A/B/C ===")
    display(comparison.ari_matrix(position).round(3))

    print(f"=== top row {position}: unseen corpus ===")
    unseen_ari[position] = pairwise_ari({variant: unseen_runs[position, variant][0] for variant in PEG_VARIANTS}, unseen)
    display(unseen_ari[position][0].round(3))

Does the disagreement depend on how many dimensions are really active? Mean ARI by the true number of active dimensions, for every pair of formulations (A/B/C pooled):

In [ ]:
ari_by_k = comparison.ari_by_k()

for position in SELECTED:
    print(f"=== top row {position}: mean ARI by k ===")
    display(ari_by_k.loc[position].unstack("k_true").rename_axis(None, axis=1).rename_axis(None).round(3))

## 6. Which dimensions and subgroups? (F, C, P)

On the unseen corpus, for every formulation: **F** is the dimension used for the splits at each depth, **C** how consistently a subgroup lands on one pole (5 most frequent), **P** the mean PEG of the split that creates a subgroup (5 highest).

In [ ]:
for (position, variant), (pipe, results) in unseen_runs.items():
    print(f"##### top row {position} | {variant}")
    display(results["F"])
    display(results["C"].head(5))
    display(results["P"].head(5))

## 7. The same text, different trees

Where do the formulations disagree most? For every text of A/B/C we take the mean ARI over all pairs of formulations; the lowest values are the texts on which the formulations build the most different trees (for the first chosen row). For each of them: the true structure, one line per formulation, and the trees themselves.

Set `TEXTS = [("A_default", 12), ...]` to look at specific texts instead.

In [ ]:
N_TEXTS = 3
TEXTS = None        # or a list of (corpus, text_id)

disagreement = comparison.disagreement(SELECTED[0])     # lowest mean ARI first
display(disagreement.head(N_TEXTS))

chosen = TEXTS or list(zip(disagreement["corpus"].head(N_TEXTS), disagreement["text_id"].head(N_TEXTS)))

for corpus, text_id in chosen:
    comparison.show_text(corpus, text_id, SELECTED[0])

## 8. Save

Everything goes into `benchmark_results/peg_comparison/`: `per_run.csv` has every number of every run and corpus, `overview.csv` the table of section 4, `recovery_by_k.csv` and `ari_by_k.csv` the by-k tables, `selected_settings.csv` the settings used, the ARI matrices and per-text values (A/B/C and the unseen corpus), and the complete F, C and P tables of the unseen corpus.

In [ ]:
comparison.save(OUT_DIR)

for position, (matrix, per_text) in unseen_ari.items():
    matrix.to_csv(OUT_DIR / f"ari_setting{position}_{INFERENCE_CORPUS}.csv")
    per_text.to_csv(OUT_DIR / f"ari_per_text_setting{position}_{INFERENCE_CORPUS}.csv", index=False)

for (position, variant), (pipe, results) in unseen_runs.items():
    for name in "FCP":
        results[name].to_csv(OUT_DIR / f"{name}_top{position}_{variant}.csv")

print("Saved to:", OUT_DIR.resolve())